# Feed Calculator: shared data preparation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/cases/feed-calculator-preparation.ipynb)

**Starter** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

The libraries used below are already available in Colab; no installation cell is needed. For local Jupyter use, see the [environment guidance](https://github.com/gromicho/teaching/blob/main/docs/SETUP.md).


## Shared data preparation
This notebook prepares the common Feed Calculator workbook for ABW and AABW. The course repository and Canvas supply the actual questions and release dates. Build your model only after identifying the meaning and units of every rule. The workbook is an educational case, not nutritional advice.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/cases/feed-calculator.xlsx"), Path("feed-calculator.xlsx")]
                 if p.is_file()), Path("feed-calculator.xlsx"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/cases/feed-calculator.xlsx"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "2e3653fe1b3a9995abedfb5902157dea2616c72550b62b7b060442e281bf8fe7":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "2e3653fe1b3a9995abedfb5902157dea2616c72550b62b7b060442e281bf8fe7", "Unexpected local data version"


In [ ]:
import pandas as pd
data_file = data_path
data = pd.read_excel(data_file,sheet_name=None)
data.keys()


In [ ]:
import pandas as pd
data = pd.read_excel(data_file, sheet_name=None)


In [ ]:
# Below, the names of the sheets are displayed
data.keys()


## Organizing your data

We can store one of the sheets in dedicated variables.

We store the ingredient database in ingredients, and so forth.

In [ ]:
ingredients      = data['Ingredient Database']
nutrient_rules   = data['Nutrient Rules']
ingredient_rules = data['Ingredient Rules']


## Inspect the data

Now we use the command '.head()' to show the first rows of DataFrames.

Without any argument, the function gives the first 5 rows of a DataFrame.

If we enter a number within the brackets, that number will be equal to the rows that are displayed.

In [ ]:
ingredients.head(10)


A pie chart of the availability shows which part of the ingredients are available and which part is not available.


In [ ]:
ingredients['Availability'].value_counts().plot(kind='pie',autopct='%1.1f%%')


We can list those that are available

In [ ]:
ingredients[ingredients.Availability].Name


To compute a summary of the statistics of the price of the ingredients, we can use .describe(). We can for example see the number of ingredients, the mean and the smallest and largest prices in the dataset.

In [ ]:
ingredients.Price.describe()


It might be interesting as well to get the statistics for the part of the ingredients that are available. How can we obtain that?

In [ ]:
ingredients[ingredients['Availability']].Price.describe()


Notice how we can access a column in different ways.
We may also be interested in knowing the names of all columns,

In [ ]:
ingredients.columns


There is a relation between the `ingredients` table and the `nutrient_rules`.

In [ ]:
data['Nutrient Rules'].Nutrient


In [ ]:
nutrients_in_rules = set( data['Nutrient Rules'].Nutrient )
columns_in_ingredients = set( ingredients.columns )

nutrients_in_rules, columns_in_ingredients


Let us focus on the nutrients that are present in the ingredients and mentioned in rules.

In [ ]:
nutrients = nutrients_in_rules & columns_in_ingredients

nutrients


In [ ]:
ingredients[ingredients.Availability][['Name']+sorted(nutrients)].plot(
    x='Name',
    kind='barh',
    stacked=True,
    figsize=(13,8)
)


## Collect the ingredients that are available together with the relevant columns

In [ ]:
available_ingredients = ingredients[ingredients.Availability][
    ['Name','Reference name','Price']+sorted(nutrients)
].set_index('Reference name')

available_ingredients


## Collect the nutrient bounds

Note that 'not available' in a data frame is not the same as None!

In [ ]:
nutrient_bounds = { nut : (lb if not pd.isna(lb) else 0,     # no lower bound translates into 0 as lower bound
                           ub if not pd.isna(ub) else None)  # no upper bound becomes None
                    for nut,lb,ub in zip(nutrient_rules.Nutrient,nutrient_rules['Lower Bound'],nutrient_rules['Upper Bound'])
                  }


In [ ]:
nutrient_bounds


## Collect the ingredient bounds

Note again that 'not available' in a data frame is not the same as None!

In [ ]:
ingredient_bounds = { ing : (lb if not pd.isna(lb) else 0,    # no lower bound translates into 0 as lower bound
                             ub if not pd.isna(ub) else None) # no upper bound becomes None
                      for ing,lb,ub in zip(ingredient_rules.Ingredient,ingredient_rules['Lower Bound'],ingredient_rules['Upper Bound']) }


In [ ]:
ingredient_bounds


## Collect the combined ingredient rules

This is slightly more complex, as we need to know where this data is placed in the sheet, hence the data frame.
Note how we use `.dropna()` on series to leave only the values defined.
Note as well how we filter the ingredients that are not available.
It may happen that a combined rule disappears, because it did only relate to not available ingredients.

If you want to know where the `Unnamed: ` columns are coming from, just examine the `ingredient_rules` data frame.


In [ ]:
combined_ingredient_rules   = []
set_of_available_ingredients = set(available_ingredients.index)
for c in ['Unnamed: '+str(i) for i in range(5,13)]:
    aux = ingredient_rules[[c]].dropna().values
    aux = [ v[0] for v in aux ]
    upperbound          = aux[0]
    ingredients_in_rule = set(aux[1:]).intersection(set_of_available_ingredients)
    if ingredients_in_rule:
        combined_ingredient_rules.append((upperbound,ingredients_in_rule))
combined_ingredient_rules


# Now optimize!

This may be a good moment to read chapter 2 of the textbook and study the notebooks of the same chapter on the [online companion](https://mobook.github.io/MO-book/notebooks/02/02.00.html) in order to express the model and solve it for these data!

Please try to solve the problem before the tutorial!

## Continue in your course
Formulate the model using the released questions in Canvas. No complete model or instructor answer is provided here.
